# SpendShield — Revised Baseline and Error Analysis

This notebook evaluates the fixed majority reference and transparent Gaussian
Naive Bayes baseline on v2, then runs the existing controlled error-analysis
workflow. The test split remains final and held out.


## Protocol

Preprocessing is train-fitted. Validation is reported for research comparison;
test metrics are not used for fitting or model selection. Metrics classify
synthetic scenarios only.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "ml").is_dir() and (candidate / "data" / "synthetic").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

V2_DIR = ROOT / "data" / "synthetic" / "v2"
V2_FEATURE_DIR = V2_DIR / "features"
V2_BASELINE_DIR = V2_DIR / "baseline"
V2_ERROR_DIR = V2_DIR / "error_analysis"
V2_COMPARISON_DIR = ROOT / "data" / "synthetic" / "comparison"
from ml.feature_matrix import build_feature_artifact
from ml.research_baseline import evaluate_research_baseline
from ml.error_analysis import run_error_analysis

if not (V2_FEATURE_DIR / "feature_manifest.json").exists():
    build_feature_artifact(V2_DIR, V2_FEATURE_DIR)
baseline = evaluate_research_baseline(
    V2_FEATURE_DIR,
    V2_BASELINE_DIR / "research_baseline_results.json",
)
analysis = run_error_analysis(
    V2_DIR,
    V2_FEATURE_DIR,
    V2_BASELINE_DIR / "research_baseline_results.json",
    V2_ERROR_DIR,
)
gaussian = baseline["models"]["gaussian_naive_bayes"]
majority = baseline["models"]["majority_class"]
print(json.dumps({
    "baseline_version": baseline["baseline_version"],
    "majority_test": majority["test"]["metrics"],
    "gaussian_validation": gaussian["validation"]["metrics"],
    "gaussian_test": gaussian["test"]["metrics"],
    "error_analysis_decision": analysis["decision"],
}, indent=2))


{
  "baseline_version": "1.1.0",
  "majority_test": {
    "accuracy": 0.725376,
    "macro_precision": 0.120896,
    "macro_recall": 0.166667,
    "macro_f1": 0.140139,
    "weighted_f1": 0.609919,
    "classes": [
      "normal",
      "synthetic_behavior_deviation",
      "synthetic_combined_pattern",
      "synthetic_high_amount",
      "synthetic_rapid_repeat",
      "synthetic_unusual_time"
    ],
    "per_class": {
      "normal": {
        "precision": 0.725376,
        "recall": 1.0,
        "f1": 0.840832,
        "support": 869
      },
      "synthetic_behavior_deviation": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "support": 50
      },
      "synthetic_combined_pattern": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "support": 24
      },
      "synthetic_high_amount": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "support": 116
      },
      "synthetic_rapid_repeat": {
    

In [2]:
test_metrics = baseline["models"]["gaussian_naive_bayes"]["test"]["metrics"]
for class_name, values in test_metrics["per_class"].items():
    print(class_name, values)
print("Test confusion matrix:")
for row in test_metrics["confusion_matrix"]:
    print(row)


normal {'precision': 0.790389, 'recall': 0.889528, 'f1': 0.837033, 'support': 869}
synthetic_behavior_deviation {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 50}
synthetic_combined_pattern {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 24}
synthetic_high_amount {'precision': 0.4, 'recall': 0.189655, 'f1': 0.25731, 'support': 116}
synthetic_rapid_repeat {'precision': 0.317881, 'recall': 1.0, 'f1': 0.482412, 'support': 48}
synthetic_unusual_time {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 91}
Test confusion matrix:
[773, 1, 5, 21, 69, 0]
[31, 0, 1, 4, 14, 0]
[13, 1, 0, 3, 7, 0]
[80, 2, 2, 22, 10, 0]
[0, 0, 0, 0, 48, 0]
[81, 0, 2, 5, 3, 0]


A lower baseline score after reducing generator shortcuts is not
automatically a dataset failure. It indicates the revised task is harder; the
readiness decision also considers overlap, class support, temporal integrity,
and leakage.
